<a href="https://colab.research.google.com/github/swayambel/Yarkovsky-induced-integrator-analysis/blob/main/yarkovsky_stability_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Impact of Yarkovsky Effect on Orbital Stability under Different Numerical Integrators

---

**Author:** Swayam Beluse  
**Subject:** Computational Physics — Orbital Dynamics & Numerical Methods  

---

> *"The Yarkovsky effect is one of the most subtle yet consequential non-gravitational forces in solar system dynamics — a thermal whisper that reshapes asteroid orbits over millennia. This study examines how different numerical integration schemes interact with such small perturbations, and when numerical error drowns out the physical signal."*

---

## 1. Project Overview

This notebook presents a structured computational study of how the **Yarkovsky effect**, a subtle radiation-pressure-induced thermal force influences the orbital evolution of a small body in a two-body gravitational system. We investigate this under three numerical integrators of varying order and symplecticity.

### Motivation

In planetary defense and long-term asteroid tracking, both **physical perturbations** (e.g., Yarkovsky drift) and **numerical errors** accumulate over time. A naive simulation might misattribute numerical drift as a real physical effect — or worse, let numerical noise completely mask a real signal. Understanding this interplay is central to reliable long-term orbital predictions.

### Scope

- Simulate a 2D two-body orbit with and without the Yarkovsky perturbation
- Compare three integrators: **Euler**, **RK4**, and **Velocity Verlet**
- Quantify numerical vs. physical drift through energy analysis
- Assess timestep sensitivity and stability
- Draw physically meaningful conclusions about integrator suitability

---

## 2. Project Objectives

The objectives of this study are:

1. Implement a normalized **two-body gravitational model** with ($GM = 1$).

2. Model the Yarkovsky effect as a small **tangential acceleration** acting along the velocity direction.

3. Implement and compare numerical integrators: **Euler** (baseline), **RK4** (high-order), and **Velocity Verlet** (symplectic).

4. Analyze **orbital trajectories** and **energy conservation** over long timescales.

5. Distinguish **numerical drift** from **physical** (Yarkovsky-induced) **drift**.

6. Perform **timestep sensitivity analysis** to evaluate accuracy and stability.

7. Determine the most suitable integrator for **long-term orbital stability under perturbations**.

---


## 3. Physical Model

### 3.1 Gravitational Two-Body System

We model a small body (e.g., asteroid) orbiting a central mass under Newtonian gravity. The system is normalized so that:

$$GM = 1, \quad r_0 = 1 \text{ AU (normalized)}, \quad T_0 = 2\pi \text{ (one orbit)}$$

The gravitational equation of motion is:

$$\frac{d^2 \mathbf{r}}{dt^2} = -\frac{GM\, \mathbf{r}}{|\mathbf{r}|^3}$$

### 3.2 The Yarkovsky Effect

The **Yarkovsky effect** arises from anisotropic thermal radiation from a rotating body. A sunlit asteroid absorbs solar radiation and re-emits it as thermal photons preferentially in a direction offset from the Sun–asteroid line due to thermal inertia. This produces a net non-gravitational force approximately tangential to the orbit.

In the diurnal Yarkovsky variant, this force acts along the velocity vector for a prograde rotator (leading to orbital expansion) or against it for a retrograde rotator (leading to contraction). We model it as a **tangential acceleration approximately aligned with the velocity vector**:

$$\mathbf{a}_{\text{Yark}} = A \cdot \hat{\mathbf{v}} = A \cdot \frac{\mathbf{v}}{|\mathbf{v}|}$$

where $A$ is a small scalar amplitude ($ 10^{-6}$ to $10^{-4}$ in normalized units).

### 3.3 Total Acceleration

$$\mathbf{a}_{\text{total}} = \mathbf{a}_{\text{grav}} + \mathbf{a}_{\text{Yark}} = -\frac{\mathbf{r}}{|\mathbf{r}|^3} + A \cdot \frac{\mathbf{v}}{|\mathbf{v}|}$$

### 3.4 Specific Orbital Energy

For a purely gravitational orbit, the specific mechanical energy is conserved:

$$E = \frac{1}{2}|\mathbf{v}|^2 - \frac{GM}{|\mathbf{r}|} = \frac{1}{2}v^2 - \frac{1}{r}$$

Deviations in $|E(t) - E_0|$ provide a direct measure of numerical error in the pure gravity case, and reflect the combined effect of **numerical error** and **physical (Yarkovsky-induced) drift** when perturbations are included.

---

## 4. Equations of Motion

Writing the system as a first-order ODE in phase space $\mathbf{y} = (x, y, v_x, v_y)$:

$$\frac{d}{dt}\begin{pmatrix} x \\ y \\ v_x \\ v_y \end{pmatrix} = \begin{pmatrix} v_x \\ v_y \\ a_x \\ a_y \end{pmatrix}$$

where:

$$a_x = -\frac{x}{(x^2+y^2)^{3/2}} + A \frac{v_x}{\sqrt{v_x^2+v_y^2}}$$

$$a_y = -\frac{y}{(x^2+y^2)^{3/2}} + A \frac{v_y}{\sqrt{v_x^2+v_y^2}}$$

The inclusion of the Yarkovsky term breaks the Hamiltonian structure of the system, rendering it **non-conservative**. As a result, the total mechanical energy is no longer conserved, and its evolution reflects a combination of **physical drift** (due to the perturbation) and **numerical error** (introduced by the integration scheme).

Distinguishing between these two contributions is central to the analysis carried out in this study.

---

## 5. Setup and Initial Conditions

This section defines the normalized physical parameters, initial conditions, and simulation timescales used throughout the study. Two simulation regimes are considered: a short-duration run for integrator comparison and a long-duration run to capture cumulative Yarkovsky-induced drift.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import LogLocator
import warnings
warnings.filterwarnings('ignore')

plt.style.use("seaborn-v0_8-darkgrid")
COLORS = {
    'euler':  '#e74c3c',
    'rk4':    '#2ecc71',
    'verlet': '#3498db',
    'ref':    '#95a5a6'
}

# --- Physical Parameters ---
GM        = 1.0          # Normalized gravitational parameter
A_YARK = 1e-8            # chosen so physical drift is comparable to numerical error

# --- Initial Conditions (slightly elliptical orbit) ---
X0  = 1.0
Y0  = 0.0
VX0 = 0.0
VY0 = 1.1                 # Slightly above circular velocity → bound elliptical orbit

STATE0 = np.array([X0, Y0, VX0, VY0])

# --- Simulation Parameters ---
DT          = 0.01
T_END_SHORT = 20 * np.pi  # ~10 orbits
T_END_LONG  = 200 * np.pi # ~100 orbits

print("=" * 55)
print("  Simulation Setup Summary")
print("=" * 55)
print(f"  GM (normalized)        : {GM}")
print(f"  Yarkovsky amplitude A  : {A_YARK:.1e}")
print(f"  Initial position (x,y) : ({X0}, {Y0})")
print(f"  Initial velocity (vx,vy): ({VX0}, {VY0})")
print(f"  Timestep dt            : {DT}")
print(f"  Short sim duration     : {T_END_SHORT:.2f} (~10 orbits)")
print(f"  Long sim duration      : {T_END_LONG:.2f} (~100 orbits)")
r0 = np.sqrt(X0**2 + Y0**2)
v0 = np.sqrt(VX0**2 + VY0**2)
E0 = 0.5 * v0**2 - GM / r0
print(f"  Initial energy E0      : {E0:.6f}")
print("=" * 55)

  Simulation Setup Summary
  GM (normalized)        : 1.0
  Yarkovsky amplitude A  : 1.0e-08
  Initial position (x,y) : (1.0, 0.0)
  Initial velocity (vx,vy): (0.0, 1.1)
  Timestep dt            : 0.01
  Short sim duration     : 62.83 (~10 orbits)
  Long sim duration      : 628.32 (~100 orbits)
  Initial energy E0      : -0.395000


### Note on Initial Conditions

With $v_y = 1.1 > v_{\text{circ}} = \sqrt{GM/r} = 1.0$, the orbit has a slightly higher specific energy than the circular case, resulting in a mildly elliptical trajectory. This choice is deliberate: a perfectly circular orbit makes drift harder to visualize, while a highly eccentric orbit introduces strong periapsis dynamics that complicate interpretation.

The chosen initial condition provides a controlled regime in which long-term orbital drift due to the Yarkovsky effect can be observed clearly without being dominated by geometric extremes.

### 6. Yarkovsky Effect Modeling

The following functions define the physical model used throughout the simulations. Gravitational acceleration provides the baseline dynamics, while a simplified Yarkovsky term introduces a small tangential perturbation aligned with the velocity vector.

The modular structure allows controlled comparison between purely gravitational motion and perturbed dynamics, enabling clear separation of **numerical drift** and **physical (Yarkovsky-induced) drift**.

In [2]:
# --- Core Physics Functions ---

def gravity(r_vec):
    r_mag = np.linalg.norm(r_vec)
    return -GM * r_vec / r_mag**3


def yarkovsky(v_vec):
    v_mag = np.linalg.norm(v_vec)
    if v_mag < 1e-12:
        return np.zeros_like(v_vec)
    return A_YARK * v_vec / v_mag


def acceleration(r_vec, v_vec, use_yarkovsky=True):
    a = gravity(r_vec)
    if use_yarkovsky:
        a += yarkovsky(v_vec)
    return a


def specific_energy(r_vec, v_vec):
    r_mag = np.linalg.norm(r_vec)
    v_mag = np.linalg.norm(v_vec)
    return 0.5 * v_mag**2 - GM / r_mag


# --- Verify physics at initial conditions ---
r0_vec = STATE0[:2]
v0_vec = STATE0[2:]

print("Physical verification at t=0:")
print(f"  Gravity      : {gravity(r0_vec)}")
print(f"  Yarkovsky    : {yarkovsky(v0_vec)}")
print(f"  Total accel  : {acceleration(r0_vec, v0_vec)}")
print(f"  Specific E   : {specific_energy(r0_vec, v0_vec):.6f}")

ratio = np.linalg.norm(yarkovsky(v0_vec)) / np.linalg.norm(gravity(r0_vec))
print(f"  Yark/Grav ratio: {ratio:.2e}")

Physical verification at t=0:
  Gravity      : [-1. -0.]
  Yarkovsky    : [0.e+00 1.e-08]
  Total accel  : [-1.e+00  1.e-08]
  Specific E   : -0.395000
  Yark/Grav ratio: 1.00e-08


### Physical Interpretation

The Yarkovsky-to-gravity ratio is of order $10^{-8}$, confirming that it acts as a **weak perturbation** — small enough to preserve the dominant gravitational dynamics, yet sufficiently large to produce measurable drift over long timescales.

In real solar system dynamics, the Yarkovsky acceleration on a ~100-meter asteroid is of order $10^{-12}$ m/s², roughly $10^{-9}$ of the Sun's gravitational pull. While our normalized model uses a slightly amplified value for visibility, it preserves the essential separation of scales between gravitational forces and small non-gravitational perturbations.

## 7. Numerical Integrators — Description

Three integration schemes are employed to probe how numerical error interacts with weak physical perturbations (Yarkovsky effect). These methods span a range of accuracy, stability, and geometric structure.

---

### 7.1 Euler Method (1st Order, Explicit)

$$\mathbf{r}_{n+1} = \mathbf{r}_n + \Delta t \, \mathbf{v}_n$$
$$\mathbf{v}_{n+1} = \mathbf{v}_n + \Delta t \, \mathbf{a}_n$$

- **Order:** $\mathcal{O}(\Delta t)$ — first order  
- **Symplectic:** No  
- **Energy behavior:** Rapid, unbounded energy growth  
- **Role in this study:** Serves as a failure case where numerical drift overwhelms any physical perturbation signal  

---

### 7.2 Runge-Kutta 4 (4th Order, Explicit)

Uses four slope evaluations per step:

$$k_1 = f(t_n, y_n), \quad k_2 = f\!\left(t_n+\tfrac{h}{2}, y_n+\tfrac{h}{2}k_1\right)$$
$$k_3 = f\!\left(t_n+\tfrac{h}{2}, y_n+\tfrac{h}{2}k_2\right), \quad k_4 = f(t_n+h, y_n+hk_3)$$
$$y_{n+1} = y_n + \frac{h}{6}(k_1 + 2k_2 + 2k_3 + k_4)$$

- **Order:** $\mathcal{O}(\Delta t^4)$ — fourth order  
- **Symplectic:** No  
- **Energy behavior:** Excellent short-term accuracy, but exhibits slow secular drift  
- **Role in this study:** Tests whether high local accuracy can mask or distort weak physical drift signals over long timescales  

---

### 7.3 Velocity Verlet (2nd Order, Symplectic)

$$\mathbf{r}_{n+1} = \mathbf{r}_n + \Delta t \, \mathbf{v}_n + \frac{\Delta t^2}{2} \mathbf{a}_n$$
$$\mathbf{v}_{n+1} = \mathbf{v}_n + \frac{\Delta t}{2}(\mathbf{a}_n + \mathbf{a}_{n+1})$$

- **Order:** $\mathcal{O}(\Delta t^2)$  
- **Symplectic:** Yes (for conservative systems)  
- **Energy behavior:** Bounded oscillations with no secular drift in pure gravity  
- **Role in this study:** Provides a structure-preserving baseline, allowing physical perturbations to be distinguished from numerical artifacts  

---

> In the presence of weak non-conservative forces, the choice of integrator determines whether observed energy changes reflect true physical drift or numerical artifacts. Symplectic methods preserve the underlying Hamiltonian structure, making them more reliable for isolating subtle long-term effects.

## 8. Implementation of Integrators

The numerical schemes described above are implemented in a modular form, separating the integration logic from the underlying physical model. Each integrator advances the phase-space state $(\mathbf{r}, \mathbf{v})$ using the shared acceleration function, ensuring a consistent comparison across methods.

A generic simulation driver is used to propagate the system over time and record diagnostic quantities such as energy and orbital radius. This design allows identical initial conditions and perturbation settings to be applied across all integrators, enabling a controlled evaluation of numerical behavior.

In [4]:
#   Integrator Implementations

def euler_step(r, v, dt, use_yarkovsky=True, A=A_YARK):
    a = acceleration(r, v, use_yarkovsky, A)
    r_new = r + dt * v
    v_new = v + dt * a
    return r_new, v_new


def rk4_step(r, v, dt, use_yarkovsky=True, A=A_YARK):
    def deriv(r_, v_):
        return v_, acceleration(r_, v_, use_yarkovsky, A)

    dr1, dv1 = deriv(r, v)
    dr2, dv2 = deriv(r + 0.5*dt*dr1, v + 0.5*dt*dv1)
    dr3, dv3 = deriv(r + 0.5*dt*dr2, v + 0.5*dt*dv2)
    dr4, dv4 = deriv(r + dt*dr3,     v + dt*dv3)

    r_new = r + (dt / 6.0) * (dr1 + 2*dr2 + 2*dr3 + dr4)
    v_new = v + (dt / 6.0) * (dv1 + 2*dv2 + 2*dv3 + dv4)
    return r_new, v_new


def verlet_step(r, v, dt, use_yarkovsky=True, A=A_YARK):
    a_n   = acceleration(r, v, use_yarkovsky, A)
    r_new = r + dt * v + 0.5 * dt**2 * a_n
    a_np1 = acceleration(r_new, v + dt * a_n, use_yarkovsky, A)
    v_new = v + 0.5 * dt * (a_n + a_np1)
    return r_new, v_new


#   Generic Simulation Driver

def simulate(integrator_fn, state0, dt, t_end, use_yarkovsky=True, A=A_YARK):

    n_steps = int(t_end / dt)

    r = state0[:2].copy()
    v = state0[2:].copy()

    t_arr  = np.zeros(n_steps + 1)
    x_arr  = np.zeros(n_steps + 1)
    y_arr  = np.zeros(n_steps + 1)
    vx_arr = np.zeros(n_steps + 1)
    vy_arr = np.zeros(n_steps + 1)
    E_arr  = np.zeros(n_steps + 1)
    r_arr  = np.zeros(n_steps + 1)

    x_arr[0], y_arr[0]   = r
    vx_arr[0], vy_arr[0] = v
    E_arr[0]  = specific_energy(r, v)
    r_arr[0]  = np.linalg.norm(r)

    for i in range(n_steps):
        r, v = integrator_fn(r, v, dt, use_yarkovsky, A)
        t_arr[i+1]  = (i + 1) * dt
        x_arr[i+1]  = r[0]
        y_arr[i+1]  = r[1]
        vx_arr[i+1] = v[0]
        vy_arr[i+1] = v[1]
        E_arr[i+1]  = specific_energy(r, v)
        r_arr[i+1]  = np.linalg.norm(r)

    return t_arr, x_arr, y_arr, vx_arr, vy_arr, E_arr, r_arr